# Better data

In [1]:
# %%
from datasets import load_dataset
import re

sources = [
    ("roneneldan/TinyStories",                None,    "train", "text"),
    ("ajibawa-2023/Children-Stories-Collection", None, "train", "text"),
]
CAPS = {
    "roneneldan/TinyStories":                   200_000,
    "ajibawa-2023/Children-Stories-Collection": 250_000,
}

def clean(text):
    filtered = "".join(c for c in text if 32 <= ord(c) <= 126)
    filtered = re.sub(r'<\w+>', '', filtered)
    filtered = re.sub(r',\s*,', ',', filtered)
    filtered = re.sub(r'\s{2,}', ' ', filtered)
    filtered = re.sub(r"\s+", " ", filtered).strip()
    return filtered

def is_good_line(line):
    if len(line) < 40:
        return False
    if line.count(',') > 20:
        return False
    if re.search(r'\d{5,}', line):
        return False
    return True

def stream_corpus(smoke_test=False):
    all_lines = []
    for name, config, split, field in sources:
        print(f"Streaming {name}...")
        cap = 2000 if smoke_test else CAPS[name]
        ds  = load_dataset(name, config, split=split, streaming=True, trust_remote_code=True)
        lines = []
        for row in ds:
            # DailyDialog rows are lists of utterances — join into one conversation
            if isinstance(row[field], list):
                raw = " ".join(row[field])
            else:
                raw = row[field]
            line = clean(raw)
            if is_good_line(line):
                lines.append(line)
            if len(lines) >= cap:
                break
        print(f"  {name}: {len(lines):,} lines collected")
        all_lines.extend(lines)
    print(f"Total: {len(all_lines):,} lines")
    return all_lines

corpus = stream_corpus()

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'roneneldan/TinyStories' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


Streaming roneneldan/TinyStories...


`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'ajibawa-2023/Children-Stories-Collection' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


  roneneldan/TinyStories: 200,000 lines collected
Streaming ajibawa-2023/Children-Stories-Collection...
  ajibawa-2023/Children-Stories-Collection: 140,197 lines collected
Total: 340,197 lines


In [2]:
# Dataset — tokenize the corpus in memory, no file needed
from tokenizers import Tokenizer
from torch.utils.data import Dataset, DataLoader
import torch

class TextDataset(Dataset):
    def __init__(self, lines, seq_len=128):
        tok = Tokenizer.from_file("tokenizer/tokenizer.json")

        print("Tokenizing...")
        all_ids = []
        for i in range(0, len(lines), 1024):
            batch = tok.encode_batch(lines[i:i+1024])
            for enc in batch:
                all_ids.extend(enc.ids)

        self.data    = torch.tensor(all_ids, dtype=torch.long)
        self.seq_len = seq_len
        print(f"Total tokens: {len(self.data):,}")

    def __len__(self):
        return len(self.data) - self.seq_len - 1

    def __getitem__(self, idx):
        x = self.data[idx:idx + self.seq_len]
        y = self.data[idx+1:idx + self.seq_len + 1]
        return x, y

dataset = TextDataset(corpus)
print(f"Dataset samples: {len(dataset):,}")

Tokenizing...
Total tokens: 107,924,140
Dataset samples: 107,924,011


## Model

In [3]:
# %%
import torch
import torch.nn as nn
import numpy as np
import math

# %%
embedding_table  = np.load("tokenizer/token_embeddings.npy")
embedding_tensor = torch.tensor(embedding_table, dtype=torch.float32)
VOCAB_SIZE     = embedding_tensor.shape[0]   # 4096
MINILM_DIM     = embedding_tensor.shape[1]   # 384
COMPRESSED_DIM = 64
N_HEADS        = 4
N_LAYERS       = 4
FFN_DIM        = 256
MAX_SEQ        = 128
print(f"Vocab size : {VOCAB_SIZE}")
print(f"MiniLM dim : {MINILM_DIM}")

# %%
class MicroLM(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("embedding_table", embedding_tensor)
        self.compressor = nn.Sequential(
            nn.Linear(MINILM_DIM, COMPRESSED_DIM),
            nn.LayerNorm(COMPRESSED_DIM),
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer=nn.TransformerEncoderLayer(
                d_model=COMPRESSED_DIM,
                nhead=N_HEADS,
                dim_feedforward=FFN_DIM,
                dropout=0.1,
                activation="gelu",
                batch_first=True,
                norm_first=True,
            ),
            num_layers=N_LAYERS,
        )
        self.norm        = nn.LayerNorm(COMPRESSED_DIM)
        self.output_head = nn.Linear(COMPRESSED_DIM, VOCAB_SIZE, bias=False)
        # ← must be inside __init__
        self.register_buffer("pos_enc", self._build_pos_enc(MAX_SEQ, COMPRESSED_DIM))

    def _build_pos_enc(self, seq_len, dim):
        pe       = torch.zeros(seq_len, dim)
        position = torch.arange(seq_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, dim, 2) * (-math.log(10000.0) / dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe

    def forward(self, token_ids):
        B, T = token_ids.shape
        x    = self.embedding_table[token_ids]
        x    = self.compressor(x)
        x    = x + self.pos_enc[:T]
        mask = nn.Transformer.generate_square_subsequent_mask(T, device=token_ids.device)
        x    = self.transformer(x, mask=mask, is_causal=True)
        x    = self.norm(x)
        return self.output_head(x)

# %%
model     = MicroLM()
model     = torch.compile(model)
total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params : {trainable:,}")
print(f"Total params     : {total:,}")
for name, p in model.named_parameters():
    if p.requires_grad:
        print(f"  {name:45s} {p.numel():>10,}")

# %%
x      = torch.randint(0, VOCAB_SIZE, (2, 32))
logits = model(x)
print(f"Input  : {x.shape}")
print(f"Output : {logits.shape}")

Vocab size : 4096
MiniLM dim : 384


/tmp/ipykernel_349025/147302778.py:29: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.transformer = nn.TransformerEncoder(


Trainable params : 486,976
Total params     : 486,976
  _orig_mod.compressor.0.weight                     24,576
  _orig_mod.compressor.0.bias                           64
  _orig_mod.compressor.1.weight                         64
  _orig_mod.compressor.1.bias                           64
  _orig_mod.transformer.layers.0.self_attn.in_proj_weight     12,288
  _orig_mod.transformer.layers.0.self_attn.in_proj_bias        192
  _orig_mod.transformer.layers.0.self_attn.out_proj.weight      4,096
  _orig_mod.transformer.layers.0.self_attn.out_proj.bias         64
  _orig_mod.transformer.layers.0.linear1.weight     16,384
  _orig_mod.transformer.layers.0.linear1.bias          256
  _orig_mod.transformer.layers.0.linear2.weight     16,384
  _orig_mod.transformer.layers.0.linear2.bias           64
  _orig_mod.transformer.layers.0.norm1.weight           64
  _orig_mod.transformer.layers.0.norm1.bias             64
  _orig_mod.transformer.layers.0.norm2.weight           64
  _orig_mod.transformer

In [5]:
import math
import os
import torch.nn.functional as F

BATCH_SIZE = 128
LR         = 3e-4
EPOCHS     = 2
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
CKPT_DIR   = "checkpoints_v3/"
CKPT_EVERY = 100000

os.makedirs(CKPT_DIR, exist_ok=True)
print(f"Training on : {DEVICE}")
print(f"Dataset size: {len(dataset):,} samples")
print(f"Steps/epoch : {len(dataset) // BATCH_SIZE:,}")

Training on : cuda
Dataset size: 107,924,011 samples
Steps/epoch : 843,156


In [6]:
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS * len(loader))
scaler = torch.amp.GradScaler()

# %%
def save_checkpoint(epoch, step, loss, scaler,  tag=None):
    state = {
        "epoch":       epoch,
        "step":        step,
        "model_state": model.state_dict(),
        "optimizer":   optimizer.state_dict(),
        "scheduler":   scheduler.state_dict(),
        "loss":        loss,
        "scaler": scaler.state_dict()
    }
    torch.save(state, os.path.join(CKPT_DIR, "latest.pt"))
    if tag:
        torch.save(state, os.path.join(CKPT_DIR, f"{tag}.pt"))
        print(f"  Checkpoint → {tag}.pt")

# %%
RESUME_FROM = os.path.join(CKPT_DIR, "latest.pt")
model.to(DEVICE)

if os.path.exists(RESUME_FROM):
    ckpt = torch.load(RESUME_FROM, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    start_epoch = ckpt["epoch"] - 1
    print(f"Resumed — epoch {ckpt['epoch']} step {ckpt['step']} loss {ckpt['loss']:.4f}")
else:
    start_epoch = 0
    print("Starting fresh")

model.train()

# %%
import time

for epoch in range(start_epoch, EPOCHS):
    total_loss = 0
    step_times = []
    t_epoch_start = time.perf_counter()

    for step, (x, y) in enumerate(loader):
        t_step_start = time.perf_counter()

        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            logits = model(x)
            loss   = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1), ignore_index=0)

        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()
        step_times.append(time.perf_counter() - t_step_start)

        if step % 200 == 0:
            avg      = total_loss / (step + 1)
            avg_ms   = sum(step_times[-200:]) / len(step_times[-200:]) * 1000
            steps_s  = 1000 / avg_ms
            elapsed  = time.perf_counter() - t_epoch_start
            remaining = (len(loader) - step) / steps_s
            print(f"Epoch {epoch+1} | Step {step:5d} | Loss {avg:.4f} | PPL {math.exp(min(avg, 20)):.1f} "
                  f"| {avg_ms:.1f} ms/step | {steps_s:.0f} steps/s "
                  f"| elapsed {elapsed/60:.1f}m | eta {remaining/60:.1f}m")

        if step % CKPT_EVERY == 0 and step > 0:
            save_checkpoint(epoch + 1, step, total_loss / (step + 1), scaler, tag=f"ckpt_ep{epoch+1}_step{step}")

    avg_loss    = total_loss / len(loader)
    epoch_time  = time.perf_counter() - t_epoch_start
    overall_mss = epoch_time / len(loader) * 1000
    save_checkpoint(epoch + 1, step, avg_loss, scaler, tag=f"ckpt_ep{epoch+1}_final")
    print(f"\nEpoch {epoch+1} done — Loss {avg_loss:.4f} | PPL {math.exp(min(avg_loss, 20)):.1f} "
          f"| {epoch_time/60:.1f} min | {overall_mss:.1f} ms/step avg\n")

Starting fresh
Epoch 1 | Step     0 | Loss 8.3748 | PPL 4336.5 | 4341.1 ms/step | 0 steps/s | elapsed 0.2m | eta 61003.2m
Epoch 1 | Step   200 | Loss 6.8274 | PPL 922.8 | 12.8 ms/step | 78 steps/s | elapsed 0.2m | eta 180.0m
Epoch 1 | Step   400 | Loss 6.2523 | PPL 519.2 | 12.2 ms/step | 82 steps/s | elapsed 0.3m | eta 171.2m
Epoch 1 | Step   600 | Loss 5.9208 | PPL 372.7 | 11.8 ms/step | 85 steps/s | elapsed 0.3m | eta 166.0m
Epoch 1 | Step   800 | Loss 5.6938 | PPL 297.0 | 11.8 ms/step | 85 steps/s | elapsed 0.3m | eta 165.8m
Epoch 1 | Step  1000 | Loss 5.5220 | PPL 250.1 | 11.9 ms/step | 84 steps/s | elapsed 0.4m | eta 166.7m
Epoch 1 | Step  1200 | Loss 5.3854 | PPL 218.2 | 11.9 ms/step | 84 steps/s | elapsed 0.4m | eta 166.3m
Epoch 1 | Step  1400 | Loss 5.2735 | PPL 195.1 | 11.9 ms/step | 84 steps/s | elapsed 0.5m | eta 166.9m
Epoch 1 | Step  1600 | Loss 5.1804 | PPL 177.8 | 11.9 ms/step | 84 steps/s | elapsed 0.5m | eta 166.2m
Epoch 1 | Step  1800 | Loss 5.1003 | PPL 164.1 | 11.9 

KeyboardInterrupt: 

In [11]:
# %%
from tokenizers import Tokenizer

tok = Tokenizer.from_file("tokenizer/tokenizer.json")
model.eval()

def generate(prompt, max_new_tokens=100, temperature=0.8, top_k=40):
    ids = tok.encode(prompt).ids
    ids = [2] + ids   # add <bos>
    x   = torch.tensor([ids], dtype=torch.long).to(DEVICE)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(x[:, -MAX_SEQ:])
            logits = logits[:, -1, :] / temperature

            # top-k sampling
            top_vals, top_idx = torch.topk(logits, top_k)
            probs    = torch.softmax(top_vals, dim=-1)
            next_id  = top_idx[0][torch.multinomial(probs, 1)]

            x = torch.cat([x, next_id.view(1,1)], dim=1)

            if next_id.item() == 3:   # <eos>
                break

    generated_ids = x[0].tolist()
    return tok.decode(generated_ids, skip_special_tokens=True)

# %%
# Test with a few prompts
prompts = [
    "The quick brown fox",
    "In the year 2025",
    "The scientists discovered",
    "Who are you ? "
]

for prompt in prompts:
    print(f"Prompt  : {prompt}")
    print(f"Output  : {generate(prompt)}")
    print()

Prompt  : The quick brown fox
Output  : The quick brown fox and went closer to the lion. It was the most beautiful waffle. Everyone was so happy to learn. They all played together in the garden and the boots were happy. They became friends and had fun. They played together every day and they had lots of fun.One day, the waffle had a very long time. The boot was happy and laughed. The boot went very happy. The boots were good friends who liked to play with their

Prompt  : In the year 2025
Output  : In the year 20250/3. Sarry, let's think about their favorite adventure." With excitement, they started exploring the wonders of the world. They discovered that the sky was a long time for them! This made their friends realize that science had been studying its beauty. Their lovely glowing downs and filling their tiny light. As they continued talking, they saw an old man talking about the mysteries. "This looks blue

Prompt  : The scientists discovered
Output  : The scientists discovered all 

In [22]:
import os
import torch

# test generation on each checkpoint
prompts = ["The quick brown fox", "Once upon a time", "Do you wanna kill me?"]

In [13]:
%%time
for f in sorted(os.listdir("checkpoints_v3/")):
    if not f.endswith(".pt"):
        continue
    ckpt = torch.load(f"checkpoints_v3/{f}", map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    
    print(f"\n{'='*50}")
    print(f"Checkpoint: {f} | Loss: {ckpt['loss']:.4f}")
    for prompt in prompts:
        print(f"\nPrompt : {prompt}")
        print(f"Output : {generate(prompt, max_new_tokens=200)}")



Checkpoint: ckpt_ep1_final.pt | Loss: 3.4378

Prompt : The quick brown fox
Output : The quick brown fox. One day, he started to cry. He was scared and sad.The dog got lost. He was getting dark and cute. He grabbed a paw and pulled the pile of duck. The dog was very sad. It was dark and they wanted to run away. He tried to bounce it. He cried and tried to hide. The dog was naughty and cried. He did not want to catch his py. He heard a loud noise. He did not hear the noise too. He tried to get hurt. He did not run. He tried to hurt his friends to see the dog. He felt sorry for the dog. He said, "I should not listen. I will not mean to you. I will not help you. I'm sorry. I will help you. I will respect it too."The dog and the frog was sorry. He kissed them. He wished he could not like the dog.

Prompt : In the year 2025
Output : In the year 202520,00035, and 9. That's okay, Timmy!But then, suddenly, Mr. Smith said, "That's because of 2104 of them!"Everyone agreed, while the first half o